# Analisis Rendemen Karkas, Efisiensi Pemotongan, dan Manajemen Piutang Lapak Ayam Potong Tradisional
**Dataset:** 1.937 catatan transaksi harian (periode 90 hari operasional, Juni - Agustus 2026).

Notebook ini memproses data operasional harian lapak ayam potong pasar tradisional, mengevaluasi rasio rendemen karkas terhadap bobot hidup (*carcass yield ratio*), mengidentifikasi margin laba berdasarkan jenis potongan, serta memetakan struktur piutang pelanggan tetap (buku bon). Seluruh visualisasi diekspor dalam format 300 DPI ke direktori `images/`.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

os.makedirs('images', exist_ok=True)
os.makedirs('data', exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.sans-serif': 'Inter',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.titlesize': 14,
    'figure.figsize': (10, 5)
})

print("Pustaka analitis siap. Direktori data dan images terverifikasi.")

In [ ]:
df = pd.read_csv('data/poultry_transactions.csv')

print("Dimensi Data:", df.shape)
print("\nInformasi Tipe Data:")
df.info()

print("\nPemeriksaan Missing Values:")
print(df.isnull().sum())

In [ ]:
yield_data = df['yield_ratio'] * 100

mean_yield = yield_data.mean()
std_yield = yield_data.std()
median_yield = yield_data.median()

# Uji normalitas Shapiro-Wilk pada sampel acak n=500
shapiro_stat, p_val = stats.shapiro(yield_data.sample(500, random_state=42))

print("Rerata Rendemen Karkas:", round(mean_yield, 2), "% (Std:", round(std_yield, 2), "%)")
print("Median Rendemen Karkas:", round(median_yield, 2), "%")
print("Uji Normalitas Shapiro-Wilk (n=500): W =", round(shapiro_stat, 4), "p-value =", round(p_val, 4))

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(yield_data, bins=25, kde=True, color='#0284c7', edgecolor='#0f172a')
plt.axvline(mean_yield, color='#e11d48', linestyle='--', linewidth=1.5, label=f"Rerata: {mean_yield:.2f}%")
plt.title('Distribusi Persentase Rendemen Karkas Ayam Potong (Yield Ratio %)', fontweight='bold')
plt.xlabel('Rendemen Karkas (%)')
plt.ylabel('Frekuensi Transaksi')
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig('images/carcass_yield_distribution.png', dpi=300)
plt.show()

In [ ]:
total_omzet = df['revenue_rp'].sum()
total_laba = df['gross_profit_rp'].sum()
total_piutang = df[df['payment_status'] == 'Belum Lunas']['revenue_rp'].sum()
piutang_ratio = (total_piutang / total_omzet) * 100

print("Total Omzet Kumulatif: Rp", f"{total_omzet:,}")
print("Total Laba Kotor: Rp", f"{total_laba:,}", "Margin:", round((total_laba/total_omzet)*100, 2), "%")
print("Total Piutang Belum Tertagih: Rp", f"{total_piutang:,}")
print("Rasio Piutang terhadap Total Omzet:", round(piutang_ratio, 2), "%")